# Students' Social Network Profile Clustering

This notebook builds an end-to-end clustering project for the Kaggle Students' Social Network Profile dataset. It covers data loading, EDA, preprocessing, skewness and outlier treatment, clustering, PCA visualization, and model comparison.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_style('whitegrid')

PROJECT_ROOT = Path().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.student_clustering import (
    elbow_method,
    load_student_data,
    plot_boxplots,
    plot_correlation_heatmap,
    plot_elbow_curve,
    plot_gender_by_cluster,
    plot_interest_by_cluster,
    plot_interest_totals,
    plot_missing_values,
    plot_model_comparison,
    plot_numeric_distributions,
    plot_pca_clusters,
    plot_social_activity_trend,
    preprocess_student_data,
    run_student_clustering_project,
    skewness_report,
    summarize_missing_values,
)


## 2. Optional Kaggle Download

Run this only if the CSV is not already present in `data/raw/`.

In [ ]:
# import subprocess
# subprocess.run(['kaggle', 'datasets', 'download', '-d', 'zabihullah18/students-social-network-profile-clustering'], check=True)
# unzip the downloaded file and place 03_Clustering_Marketing.csv in data/raw/


## 3. Load Data

In [ ]:
student_path = PROJECT_ROOT / 'data' / 'raw' / '03_Clustering_Marketing.csv'
if not student_path.exists():
    raise FileNotFoundError(f'Missing dataset: {student_path}')

df = load_student_data(str(student_path))
df.head()

In [ ]:
print('Shape:', df.shape)
df.info()

In [ ]:
summarize_missing_values(df).head(20)

## 4. EDA and Visualizations

In [ ]:
df.describe(include='all').T.head(15)

In [ ]:
plot_missing_values(df)
plot_numeric_distributions(df, ['age', 'numberoffriends', 'gradyear'])
plot_boxplots(df, ['age', 'numberoffriends'])
plot_correlation_heatmap(df)
plot_interest_totals(df, top_n=10)

In [ ]:
if 'gender' in df.columns:
    plt.figure(figsize=(6, 4))
    sns.countplot(data=df, x='gender', order=df['gender'].fillna('Missing').value_counts().index)
    plt.title('Gender Distribution')
    plt.tight_layout()
    plt.show()

## 5. Skewness Check and Transformation Logic

In [ ]:
skew_report = skewness_report(df)
skew_report.head(20)

In [ ]:
high_skew = skew_report[skew_report.abs() > 1]
print('Highly skewed variables:')
print(high_skew)

The preprocessing pipeline below handles:

- missing value imputation
- IQR-based outlier capping
- Yeo-Johnson power transformation for skew reduction
- standardization
- one-hot encoding for gender

In [ ]:
cleaned_df, features, feature_names, transformer = preprocess_student_data(df)
print('Feature matrix shape:', features.shape)
pd.Series(feature_names).head(15)

## 6. Elbow Method

In [ ]:
elbow_df = elbow_method(features)
elbow_df

In [ ]:
plot_elbow_curve(elbow_df)

## 7. Build Clustering Models

In [ ]:
artifacts = run_student_clustering_project(str(student_path), n_clusters=3)
artifacts.model_scores

In [ ]:
artifacts.cleaned_data.head()

## 8. PCA Visualization

In [ ]:
plot_pca_clusters(artifacts.pca_components, artifacts.kmeans_labels, title='K-Means Clusters in PCA Space')

## 9. Cluster Profiling

In [ ]:
artifacts.cluster_profile

In [ ]:
plot_gender_by_cluster(artifacts.cleaned_data)
plot_interest_by_cluster(artifacts.cleaned_data)
plot_social_activity_trend(artifacts.cleaned_data)

## 10. Compare K-Means, Hierarchical, and DBSCAN

In [ ]:
artifacts.model_scores

In [ ]:
plot_model_comparison(artifacts.model_scores)

## 11. Conclusion

This project segments students based on demographics and interest patterns. PCA helps visualize the clusters in two dimensions, while silhouette score helps compare clustering quality across different algorithms.